In [1]:
# ============================================================
# CELL 1: Imports and Setup
# ============================================================
# Why these libraries?
# - pandas: our main tool for working with tabular data
# - numpy: mathematical operations
# - matplotlib/seaborn: visualizations to understand the data
# - warnings: suppress non-critical warnings for cleaner output

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# This ensures our charts look clean in the notebook
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Reproducibility — always set a random seed in DS projects
# This means if you run the code again, you get identical results
# Interviewers love this — it shows you understand reproducibility
RANDOM_SEED = 42

print("Libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries loaded successfully
Pandas version: 2.3.3
NumPy version: 2.4.6


In [2]:
# ============================================================
# CELL 2: Load Raw Data
# ============================================================
# Why specify encoding?
# The dataset contains special characters (£ symbol, accented letters)
# Without the right encoding, pandas throws an error or corrupts text

RAW_DATA_PATH = "../data/raw/online_retail_II.csv"

df_raw = pd.read_csv(
    RAW_DATA_PATH,
    encoding='utf-8',        # handles special characters
    dtype={'Customer ID': str}  # load Customer ID as string, not float
                                # avoids pandas converting 12345.0 → "12345.0"
)

# Always work on a copy — never modify raw data directly
# Raw data is sacred — you should always be able to go back to it
df = df_raw.copy()

print(f"Raw dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
df.head(3)

Raw dataset shape: (1067371, 8)

Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

First 3 rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom


In [3]:
# ============================================================
# CELL 3: Pre-Cleaning Snapshot
# ============================================================
# Why do this? So we can report exactly how much data we removed
# and why. This is what you show in your portfolio and interviews.
# "We started with X rows and ended with Y rows after cleaning"

print("=" * 55)
print("PRE-CLEANING SNAPSHOT")
print("=" * 55)
print(f"Total rows:          {len(df):>10,}")
print(f"Total columns:       {df.shape[1]:>10}")
print(f"Duplicate rows:      {df.duplicated().sum():>10,}")
print(f"Missing Customer ID: {df['Customer ID'].isna().sum():>10,}")
print(f"Missing Description: {df['Description'].isna().sum():>10,}")

# Flag cancellations (invoices starting with 'C')
df['Invoice'] = df['Invoice'].astype(str)
cancellations = df[df['Invoice'].str.startswith('C')]
print(f"Cancellation rows:   {len(cancellations):>10,}")

# Negative/zero quantities and prices
print(f"Quantity <= 0 rows:  {(df['Quantity'] <= 0).sum():>10,}")
print(f"Price <= 0 rows:     {(df['Price'] <= 0).sum():>10,}")
print("=" * 55)

# Track rows removed at each step — for your portfolio report
cleaning_log = {}
cleaning_log['00_raw'] = len(df)

PRE-CLEANING SNAPSHOT
Total rows:           1,067,371
Total columns:                8
Duplicate rows:          34,335
Missing Customer ID:    243,007
Missing Description:      4,382
Cancellation rows:       19,494
Quantity <= 0 rows:      22,950
Price <= 0 rows:          6,207


In [4]:
# ============================================================
# CELL 4: Data Cleaning — Step by Step
# ============================================================

# --- STEP 1: Remove duplicate rows ---
# Why: Exact duplicates are almost certainly system errors
# (same order logged twice). They inflate purchase counts.
df.drop_duplicates(inplace=True)
cleaning_log['01_after_dedup'] = len(df)
print(f"Step 1 — Removed duplicates: {cleaning_log['00_raw'] - cleaning_log['01_after_dedup']:,} rows removed")

# --- STEP 2: Separate cancellations BEFORE dropping ---
# Why: We don't want returns in our revenue/RFM calculations
# BUT we want to use them as a behavioral feature later
# (customers who return a lot are churn risks)
cancellations_df = df[df['Invoice'].str.startswith('C')].copy()
df = df[~df['Invoice'].str.startswith('C')]
cleaning_log['02_after_cancel_removal'] = len(df)
print(f"Step 2 — Separated cancellations: {len(cancellations_df):,} rows saved separately")

# --- STEP 3: Remove rows with missing Customer ID ---
# Why: We cannot do ANY customer-level analysis without knowing
# who the customer is. These rows are useless for modeling.
# We already decided this in our planning session.
df = df[df['Customer ID'].notna()]
cleaning_log['03_after_null_customerid'] = len(df)
print(f"Step 3 — Removed missing Customer IDs: {cleaning_log['02_after_cancel_removal'] - cleaning_log['03_after_null_customerid']:,} rows removed")

# --- STEP 4: Remove invalid quantities ---
# Why: Quantity of 0 or negative (that aren't cancellations)
# are data entry errors. You can't buy -5 items.
df = df[df['Quantity'] > 0]
cleaning_log['04_after_quantity'] = len(df)
print(f"Step 4 — Removed invalid quantities: {cleaning_log['03_after_null_customerid'] - cleaning_log['04_after_quantity']:,} rows removed")

# --- STEP 5: Remove invalid prices ---
# Why: Price of 0 or negative makes no business sense
# for a retail transaction
df = df[df['Price'] > 0]
cleaning_log['05_after_price'] = len(df)
print(f"Step 5 — Removed invalid prices: {cleaning_log['04_after_quantity'] - cleaning_log['05_after_price']:,} rows removed")

# --- STEP 6: Parse dates ---
# Why: Dates loaded as strings can't be used for time calculations
# We need this for RFM (Recency) and time-based features
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"Step 6 — Dates parsed successfully")

# --- STEP 7: Engineer TotalPrice column ---
# Why: Revenue per line item = Quantity × Price
# This is the foundation of all monetary calculations
df['TotalPrice'] = df['Quantity'] * df['Price']
print(f"Step 7 — TotalPrice column created")

# --- STEP 8: Clean Customer ID format ---
# Remove decimal points if any (e.g. "12345.0" → "12345")
df['Customer ID'] = df['Customer ID'].str.replace('.0', '', regex=False).str.strip()
print(f"Step 8 — Customer ID format cleaned")

print(f"\nFinal clean dataset shape: {df.shape}")

Step 1 — Removed duplicates: 34,335 rows removed
Step 2 — Separated cancellations: 19,104 rows saved separately
Step 3 — Removed missing Customer IDs: 234,437 rows removed
Step 4 — Removed invalid quantities: 0 rows removed
Step 5 — Removed invalid prices: 70 rows removed
Step 6 — Dates parsed successfully
Step 7 — TotalPrice column created
Step 8 — Customer ID format cleaned

Final clean dataset shape: (779425, 9)


In [5]:
# ============================================================
# CELL 5: Cleaning Summary — Your Portfolio Evidence
# ============================================================
# This is the table you screenshot for your README and portfolio

print("=" * 55)
print("CLEANING SUMMARY REPORT")
print("=" * 55)
steps = {
    "Raw data":                  cleaning_log['00_raw'],
    "After deduplication":       cleaning_log['01_after_dedup'],
    "After removing returns":    cleaning_log['02_after_cancel_removal'],
    "After removing null IDs":   cleaning_log['03_after_null_customerid'],
    "After removing qty <= 0":   cleaning_log['04_after_quantity'],
    "After removing price <= 0": cleaning_log['05_after_price'],
}

for step, count in steps.items():
    print(f"  {step:<30} {count:>10,} rows")

total_removed = cleaning_log['00_raw'] - cleaning_log['05_after_price']
pct_retained = (cleaning_log['05_after_price'] / cleaning_log['00_raw']) * 100

print("=" * 55)
print(f"  Total rows removed:  {total_removed:>10,}")
print(f"  Rows retained:       {cleaning_log['05_after_price']:>10,} ({pct_retained:.1f}%)")
print(f"  Cancellations saved: {len(cancellations_df):>10,} (for feature engineering)")
print("=" * 55)

CLEANING SUMMARY REPORT
  Raw data                        1,067,371 rows
  After deduplication             1,033,036 rows
  After removing returns          1,013,932 rows
  After removing null IDs           779,495 rows
  After removing qty <= 0           779,495 rows
  After removing price <= 0         779,425 rows
  Total rows removed:     287,946
  Rows retained:          779,425 (73.0%)
  Cancellations saved:     19,104 (for feature engineering)


In [6]:
# ============================================================
# CELL 6: Save Cleaned Data
# ============================================================
# Why save as CSV? Universal format, readable by any tool
# We save to data/processed/ — never overwrite raw data

os.makedirs('../data/processed', exist_ok=True)

# Save main clean dataset
df.to_csv('../data/processed/clean_transactions.csv', index=False)

# Save cancellations separately — we'll use these later
cancellations_df.to_csv('../data/processed/cancellations.csv', index=False)

print("Clean dataset saved to:      data/processed/clean_transactions.csv")
print("Cancellations saved to:      data/processed/cancellations.csv")
print(f"\nClean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range: {df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")
print(f"Unique customers: {df['Customer ID'].nunique():,}")
print(f"Unique products:  {df['StockCode'].nunique():,}")
print(f"Countries:        {df['Country'].nunique()}")

Clean dataset saved to:      data/processed/clean_transactions.csv
Cancellations saved to:      data/processed/cancellations.csv

Clean dataset: 779,425 rows × 9 columns
Date range: 2009-12-01 to 2011-12-09
Unique customers: 5,878
Unique products:  4,631
Countries:        41
